In [1]:
#Import Libraries
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
#Load dataset
data_path = Path("..") / "data" / "raw data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)

#Minor clean up

#Drop non-predictive column
df = df.drop("customerID", axis=1)

#Convert target to numeric
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df.head()




In [ ]:
#Handle Categorical Encoding

#Separate categorical and numeric columns
cat_cols = df.select_dtypes("object").columns
num_cols = df.select_dtypes(exclude="object").columns

#One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

#Confirm new shape
print("Encoded shape:", df_encoded.shape)


In [ ]:
#Train, Test & Split

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
#Feature Scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
#Baseline: Logistic Regression Model

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

RocCurveDisplay.from_estimator(log_reg, X_test_scaled, y_test)
plt.title("ROC Curve - Logistic Regression")
plt.show()


In [ ]:
#Ensemble: Random Forest Model

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

RocCurveDisplay.from_estimator(rf, X_test, y_test)
plt.title("ROC Curve - Random Forest")
plt.show()


In [ ]:
#Gradient Boost: XGBoost

xgb = XGBClassifier(eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

RocCurveDisplay.from_estimator(xgb, X_test, y_test)
plt.title("ROC Curve - XGBoost")
plt.show()


In [ ]:
#Compare Model Performance 

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb),
    ]
})

results.sort_values(by="ROC-AUC", ascending=False, inplace=True)
print(results)
sns.barplot(x="ROC-AUC", y="Model", data=results, palette="Blues_d")
plt.title("Model Comparison by ROC-AUC")
plt.show()


In [ ]:
#Feature Importance (Tree Models)

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
sns.barplot(x=top_features, y=top_features.index, palette="Blues_r")
plt.title("Top 15 Features Driving Churn (Random Forest)")
plt.xlabel("Feature Importance Score")
plt.show()


Summary: 

The dataset consists of roughly 26.5% customer who have churned and 73.5% customers who have not, causing models to lean towards predicting no-churn.This is due to class imbalnance. 

Tenure is the greatest churn driving varilable, followed by MonthlyCharges and Electronic payment method. 

Interpretation:

Random Forest achieved the highest ROC-AUC (0.83), showing strong predictive power even without balancing.

XGBoost and Logistic Regression followed closely, indicating all models performed reasonably well.

Because the dataset has many more non-churn customers than churned ones, the models tend to predict “no churn” most of the time. To fix this, I’ll apply SMOTE, which creates new, similar examples of churned customers. This helps the models learn patterns from both groups more evenly and should improve its ability to correctly identify customers who are likely to leave.

In [ ]:
#Apply SMOTE (Synthetic Minority Oversampling)

#Initialize SMOTE
sm = SMOTE(random_state=42)

#Apply it only to training data
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_res.value_counts().to_dict())

In [ ]:
#Retrain Logistic Regression on resampled data
log_reg_sm = LogisticRegression(max_iter=1000, random_state=42)
log_reg_sm.fit(X_train_res, y_train_res)

#Predict on original test set
y_pred_lr_sm = log_reg_sm.predict(X_test)
y_prob_lr_sm = log_reg_sm.predict_proba(X_test)[:, 1]

print("Logistic Regression (SMOTE) Results:")
print(classification_report(y_test, y_pred_lr_sm, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr_sm), 3))


RocCurveDisplay.from_estimator(log_reg_sm, X_test, y_test)
plt.title("ROC Curve - Logistic Regression (SMOTE)")
plt.show()

In [ ]:
#Retrain Random Forest Classifier

rf_sm = RandomForestClassifier(n_estimators=300, random_state=42)
rf_sm.fit(X_train_res, y_train_res)

y_pred_rf_sm = rf_sm.predict(X_test)
y_prob_rf_sm = rf_sm.predict_proba(X_test)[:, 1]

print("Random Forest (SMOTE) Results:")
print(classification_report(y_test, y_pred_rf_sm, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr_sm), 3))

RocCurveDisplay.from_estimator(rf_sm, X_test, y_test)
plt.title("ROC Curve - Random Forest (SMOTE)")
plt.show()


In [ ]:
#Retrain XGBoost 

xgb_sm = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9
)

xgb_sm.fit(X_train_res, y_train_res)

y_pred_xgb_sm = xgb_sm.predict(X_test)
y_prob_xgb_sm = xgb_sm.predict_proba(X_test)[:, 1]

print("XGBoost (SMOTE) Results:")
print(classification_report(y_test, y_pred_xgb_sm, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr_sm), 3))

RocCurveDisplay.from_estimator(xgb_sm, X_test, y_test)
plt.title("ROC Curve - XGBoost (SMOTE)")
plt.show()


In [ ]:
#Prepare Model Performance after SMOTE

results_smote = pd.DataFrame({
    "Model": ["Logistic Regression (SMOTE)", "Random Forest (SMOTE)", "XGBoost (SMOTE)"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr_sm),
        roc_auc_score(y_test, y_prob_rf_sm),
        roc_auc_score(y_test, y_prob_xgb_sm)
    ]
})

results_smote.sort_values(by="ROC-AUC", ascending=False, inplace=True)
print(results_smote)

sns.barplot(x="ROC-AUC", y="Model", data=results_smote, palette="Blues_d")
plt.title("Model Comparison After SMOTE Balancing")
plt.show()


Class Imbalance Check
Before SMOTE:
{0: 4139, 1: 1495} roughly 26.5% churners.

After SMOTE:
{0: 4139, 1: 4139} perfectly balanced 50/50 classes.

Summary:
Originally, the dataset was heavily skewed toward non-churners.
By applying SMOTE, synthetic churn cases were made to create an equal class distribution.
This helps models better learn the patterns associated with customers who actually churn, improving their ability to identify churners (recall).

Model Performance Interpretation:

The models are able to distinguish churners from non-churners with approximately 0.83 ROC-AUC, demonstrating strong predictive performance for a customer retention problem.

While SMOTE balancing slightly improved overall recall and sensitivity, the gains were modest. This suggests that the dataset is already informative and that further improvements are more likely to come from model optimization rather than additional resampling.

In [ ]:
#Model optimization

#Hyperparamater tuning with RandomizedSearchCV

#Tune XGBoost first as it had best performance after SMOTE

#Split original data again
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Apply SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

#Check new class balance
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_sm.value_counts().to_dict())

#Define parameter grid
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3],
    'scale_pos_weight': [1, 2, 3]  #helps class imbalance
}

#Initialize model
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

#Randomized search setup
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,  # number of random combinations to test
    scoring='roc_auc',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

#Fit on SMOTE-balanced data
random_search.fit(X_train_sm, y_train_sm)

print("Best Parameters:", random_search.best_params_)
print("Best ROC-AUC:", random_search.best_score_)


In [ ]:
#Determine Best XGBoost on test set

best_xgb = random_search.best_estimator_
y_prob_best = best_xgb.predict_proba(X_test)[:, 1]
y_pred_05   = (y_prob_best >= 0.50).astype(int)

print("Best params:", random_search.best_params_)
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_prob_best), 3))
print(classification_report(y_test, y_pred_05, digits=3))

RocCurveDisplay.from_estimator(best_xgb, X_test, y_test)
plt.title("ROC Curve — Tuned XGBoost")
plt.show()

In [ ]:
#Optimize Model Threshold

import numpy as np
from sklearn.metrics import precision_recall_curve, confusion_matrix

precision, recall, thresholds = precision_recall_curve(y_test, y_prob_best)


#Option A: Maximize F1
f1 = 2 * precision * recall / (precision + recall + 1e-9)
thr_f1 = thresholds[np.nanargmax(f1)]
print("F1-optimal threshold:", round(thr_f1, 3))

#Option B: Target a recall (e.g., >= 0.80) with best precision
target = 0.80
mask = recall[:-1] >= target
thr_recall = thresholds[mask][np.argmax(precision[:-1][mask])] if mask.any() else 0.50
print("Threshold for recall ≥ 0.80:", round(thr_recall, 3))

def evaluate_at(th):
    y_pred = (y_prob_best >= th).astype(int)
    print(f"\n=== Threshold = {th:.3f} ===")
    print(classification_report(y_test, y_pred, digits=3))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

evaluate_at(thr_f1)
evaluate_at(thr_recall)


In [ ]:
importances = pd.Series(best_xgb.feature_importances_, index=X_test.columns)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
sns.barplot(x=top15, y=top15.index)
plt.title("Top 15 Features — Tuned XGBoost")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
from joblib import dump
from pathlib import Path
import json

models_dir = Path("..") / "models"
models_dir.mkdir(parents=True, exist_ok=True)

dump(best_xgb, models_dir / "xgb_churn_tuned.joblib")
dump(list(X_test.columns), models_dir / "feature_columns.joblib")

# pick one: thr_f1 or thr_recall based on your goal
chosen_threshold = float(thr_recall)
with open(models_dir / "serving_config.json", "w") as f:
    json.dump({
        "threshold": chosen_threshold,
        "metric": "ROC-AUC",
        "roc_auc": round(roc_auc_score(y_test, y_prob_best), 4)
    }, f)

print("Saved tuned model and config.")


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_base = RandomForestClassifier(random_state=42)

rf_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 6, 10, 14],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_grid,
    n_iter=25,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train_sm, y_train_sm)
best_rf = rf_search.best_estimator_
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]
print("RF best params:", rf_search.best_params_)
print("RF Test ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 3))
